# Simplicio 27B · Serving (vLLM + LoRA + Cloudflare Tunnel)
Runtime > Change runtime type > **GPU (A100 recomendado)**. Depois: Run all. A URL do túnel é registrada sozinha em `https://simpleti.com.br/v1`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q vllm bitsandbytes
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
import subprocess, time, re, json, urllib.request, os
# Base Qwen3.8-27B (4-bit) + adapter LoRA Simplicio-27B
cmd = ["vllm","serve","unsloth/Qwen3.8-27B-unsloth-bnb-4bit","--quantization","bitsandbytes","--load-format","bitsandbytes",
       "--enable-lora","--lora-modules","simplicio-27b=wesleysimplicio/Simplicio-27B","--max-lora-rank","64",
       "--host","0.0.0.0","--port","8000","--max-model-len","16384","--gpu-memory-utilization","0.92","--served-model-name","base-27b"]
log = open("vllm.log","w"); vllm = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=3); print("vLLM pronto"); break
    except Exception: time.sleep(10)
else:
    print(open("vllm.log").read()[-3000:]); raise SystemExit("vLLM não subiu")

In [ ]:
tun = subprocess.Popen(["./cloudflared","tunnel","--url","http://localhost:8000"], stderr=subprocess.PIPE, text=True)
url=None; t=time.time()
while time.time()-t<60:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', tun.stderr.readline())
    if m: url=m.group(0); break
print("Túnel:", url)
KEY = os.environ.get("SIMPLETI_ADMIN_KEY") or input("Chave admin do gateway: ")
req = urllib.request.Request("https://simpleti.com.br/api/set_upstream.php?key="+KEY, data=json.dumps({"upstream_url":url}).encode(),
      headers={"Content-Type":"application/json","User-Agent":"SimpleTI-Worker/1.0"})
print(urllib.request.urlopen(req, timeout=15).read().decode())

In [ ]:
import time
while True:
    time.sleep(60)  # mantém a sessão viva